# Cyanite starter: the four endpoints (with audio)

A runnable tour of the four Cyanite API endpoints you need for the challenge. Each section maps
to a guide PDF in [`../guides/`](../guides/):

1. **Get inferred AI Models for a Library Track** (tagging): `GET /v1/library-tracks/{id}/models`
2. **Find Library Tracks based on a text prompt** (free-text search): `POST /v1/private-alpha/library-tracks/prompt-search`
3. **Find Similar Library Tracks** (single seed): `POST /v1/private-alpha/library-tracks/{id}/similar`
4. **Find Similar Library Tracks (Multi-Track)** (up to 10 seeds): `POST /v1/private-alpha/library-tracks/similar`

Search returns ranked tracks with a relevance `score`; fetch a result's tags via endpoint 1 and
explain the match. Inline **audio players** (public Jamendo MP3s) are shown for results. Tag
reference: [../guides/model_outputs.md](../guides/model_outputs.md),
[../guides/tag_vocabularies.md](../guides/tag_vocabularies.md).

> Search is **private alpha** (under `/private-alpha/`); shapes may change. Mind the shared rate
> limits and quotas (see the README).

**Setup:** `pip install requests python-dotenv`; put `CYANITE_API_KEY` in a `.env` (git-ignored).


In [ ]:
import os
import csv
import json
import tempfile
import requests
import IPython.display as ipd
from dotenv import load_dotenv

load_dotenv()  # reads CYANITE_API_KEY from .env
API_KEY = os.environ["CYANITE_API_KEY"]
BASE_URL = "https://rest-api.cyanite.ai/v1"

session = requests.Session()
session.headers.update({"x-api-key": API_KEY})
print("ready")


## Pick tracks from the data pack

`data/tracks.csv` carries both ids: `track_id` (Jamendo) and `cyanite_id` (`libtr_...`, what the
API uses). We load a Jamendo -> Cyanite map and grab a few sample Cyanite ids to play with.


In [ ]:
def _tracks_path():
    for p in ("../data/tracks.csv", "data/tracks.csv"):
        if os.path.exists(p):
            return p
    raise FileNotFoundError("data/tracks.csv not found; run from the repo root or notebooks/")

JAMENDO_TO_CYANITE = {}
with open(_tracks_path()) as f:
    for row in csv.DictReader(f):
        JAMENDO_TO_CYANITE[row["track_id"]] = row["cyanite_id"]

def resolve_track_id(jamendo_id):
    # data-pack Jamendo id -> Cyanite track id
    return JAMENDO_TO_CYANITE[str(jamendo_id)]

CYANITE_IDS = list(JAMENDO_TO_CYANITE.values())
SAMPLE_TRACK_ID = CYANITE_IDS[0]
SAMPLE_TRACK_IDS = CYANITE_IDS[:3]  # a few seeds for multi-track similarity
print(len(JAMENDO_TO_CYANITE), "tracks loaded; sample cyanite id:", SAMPLE_TRACK_ID)


## Listen to a track

`play()` renders an inline audio player from the public Jamendo MP3 (downloaded and cached in
`/tmp`). It accepts a Jamendo id, a Cyanite id (`libtr_...`), or a search-result item/track (whose
`title` is the Jamendo filename, so any result is playable). It displays itself, so it works in
loops too.


In [ ]:
CYANITE_TO_JAMENDO = {c: j for j, c in JAMENDO_TO_CYANITE.items()}
AUDIO_DIR = os.path.join(tempfile.gettempdir(), "cyanite_audio")
os.makedirs(AUDIO_DIR, exist_ok=True)

def jamendo_mp3_url(jamendo_id):
    return f"https://prod-1.storage.jamendo.com/download/track/{jamendo_id}/mp32/"

def to_jamendo_id(track):
    # accepts a Jamendo id, a Cyanite id, a track dict, or a search item {track, score}
    if isinstance(track, dict):
        if isinstance(track.get("track"), dict):
            track = track["track"]                         # search item -> inner track
        if track.get("title"):
            return str(track["title"]).split(".mp3")[0]    # title is the Jamendo filename
        if track.get("externalId"):
            return str(track["externalId"])
        track = track.get("id")
    track = str(track)
    if track.startswith("libtr_"):
        return CYANITE_TO_JAMENDO.get(track)               # data-pack tracks only
    return track

def play(track, title=None):
    # render an inline audio player (downloads the public Jamendo MP3, cached)
    jam = to_jamendo_id(track)
    if not jam:
        print("No Jamendo audio for", track)
        return
    if title:
        print(title)
    path = os.path.join(AUDIO_DIR, f"{jam}.mp3")
    if not os.path.exists(path):
        r = requests.get(jamendo_mp3_url(jam), timeout=60)
        r.raise_for_status()
        with open(path, "wb") as fh:
            fh.write(r.content)
    ipd.display(ipd.Audio(filename=path))

play(SAMPLE_TRACK_ID)


## 1. Get inferred AI Models for a Library Track

`GET /v1/library-tracks/{id}/models?model=...` returns the chosen model outputs (the tags). The
response is `{"items": [ <model output>, ... ]}`. See
[the guide PDF](../guides/Get%20inferred%20AI%20Models%20for%20a%20Library%20Track.pdf),
[model_outputs.md](../guides/model_outputs.md), and [tag_vocabularies.md](../guides/tag_vocabularies.md).


In [ ]:
DEFAULT_MODELS = [
    "MainGenreV2", "SubgenreV2", "FreeGenreV3",
    "MoodSimpleV2", "MoodAdvancedV2",
    "InstrumentsV2", "CharacterV2", "MovementV2",
    "BpmV2", "TempoV1", "KeyV2", "TimeSignatureV2",
    "ValenceArousalV2", "MusicalEraV2", "MusicForV1",
    "VocalsV2", "AutoDescriptionV2", "RepresentativeSegmentV2",
]

def get_model_outputs(track_id, models=None):
    # GET /library-tracks/{id}/models -> {"items": [ <model output>, ... ]}
    models = models or DEFAULT_MODELS
    params = [("model", m) for m in models]
    resp = session.get(f"{BASE_URL}/library-tracks/{track_id}/models", params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()

outputs = get_model_outputs(SAMPLE_TRACK_ID, ["MainGenreV2", "MoodSimpleV2", "InstrumentsV2", "BpmV2", "AutoDescriptionV2"])
print(json.dumps(outputs, indent=2)[:2500])


### Summarize the tags

Pull the human-readable bits out for "why this track" explanations.


In [ ]:
def iter_model_outputs(outputs):
    # the models endpoint returns {"items": [...]}; be tolerant of other envelopes too
    if isinstance(outputs, dict):
        for key in ("items", "data", "models", "results"):
            if isinstance(outputs.get(key), list):
                yield from outputs[key]
                return
        for v in outputs.values():
            if isinstance(v, dict) and "version" in v:
                yield v
        return
    if isinstance(outputs, list):
        yield from outputs

def summarize(outputs):
    # compact dominant tags / values per model
    summary = {}
    for mo in iter_model_outputs(outputs):
        version = mo.get("version", "?")
        if mo.get("tags") is not None:
            summary[version] = mo["tags"]
        elif "tag" in mo:
            summary[version] = mo["tag"]
        elif "description" in mo:
            summary[version] = mo["description"]
    return summary

summarize(outputs)


## 2. Find Library Tracks based on a text prompt

`POST /v1/private-alpha/library-tracks/prompt-search`, body `{"query": "...", "metadataFilter": {...}}`.
Returns `{"items": [{"track": {...}, "score": 0..1}], "pageInfo": {...}}`, ordered by relevance.
`limit` is 1-50 (default 20). See
[the guide PDF](../guides/Find%20Library%20Tracks%20based%20on%20a%20text%20prompt.pdf).

`show_results()` prints each result's `score`, id and title, then plays the top few.


In [ ]:
def search_by_prompt(query, limit=20, metadata_filter=None):
    # POST /private-alpha/library-tracks/prompt-search -> {items: [{track, score}], pageInfo}
    body = {"query": query}
    if metadata_filter:
        body["metadataFilter"] = metadata_filter
    resp = session.post(
        f"{BASE_URL}/private-alpha/library-tracks/prompt-search",
        params={"limit": limit}, json=body, timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

def show_results(res, list_n=10, play_n=3):
    # print score + id + title for each result, then render players for the top few
    items = res.get("items", [])
    for it in items[:list_n]:
        track = it.get("track", it)
        score = it.get("score")
        score_str = f"{score:.3f}  " if isinstance(score, (int, float)) else ""
        print(f"{score_str}{track.get('id')}  |  {track.get('title')}")
    print("nextCursor:", res.get("pageInfo", {}).get("nextCursor"))
    for it in items[:play_n]:
        play(it.get("track", it))


In [ ]:
res = search_by_prompt("dreamy ambient piano, slow and emotional", limit=2)
print(len(res["items"]))


In [ ]:
def search_by_prompt(query, limit=20, metadata_filter=None):
    body = {"query": query}
    if metadata_filter:
        body["metadataFilter"] = metadata_filter
    resp = session.post(
        f"{BASE_URL}/private-alpha/library-tracks/prompt-search",
        params={"limit": limit}, json=body, timeout=60,
    )
    resp.raise_for_status()
    data = resp.json()

    # --- diagnostics ---
    print("REQUEST URL :", resp.request.url)          # is ?limit=N actually there?
    print("LIMIT SENT  :", limit)
    print("ITEMS BACK  :", len(data.get("items", [])))
    print("nextCursor  :", data.get("pageInfo", {}).get("nextCursor"))
    # --------------------

    return data

search_by_prompt("dreamy ambient piano, slow and emotional", limit=2)


In [ ]:
# narrow with a metadata filter (e.g. BPM range). operators: $gte $lte $eq $in
res = search_by_prompt(
    "energetic workout music",
    limit=10,
    metadata_filter={"BpmV2.tag": {"$gte": 120, "$lte": 140}},
)
show_results(res)


## 3. Find Similar Library Tracks

`POST /v1/private-alpha/library-tracks/{id}/similar`, body `{}` or `{"metadataFilter": {...}}`.
Same `{items, pageInfo}` shape, ranked by similarity to the one seed track. See
[the guide PDF](../guides/Find%20Similar%20Library%20Tracks.pdf).


In [ ]:
def find_similar(track_id, limit=20, metadata_filter=None):
    # POST /private-alpha/library-tracks/{id}/similar -> {items: [{track, score}], pageInfo}
    body = {}
    if metadata_filter:
        body["metadataFilter"] = metadata_filter
    resp = session.post(
        f"{BASE_URL}/private-alpha/library-tracks/{track_id}/similar",
        params={"limit": limit}, json=body, timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

print("seed:", SAMPLE_TRACK_ID)
res = find_similar(SAMPLE_TRACK_ID, limit=10)
show_results(res)


## 4. Find Similar Library Tracks (Multi-Track)

`POST /v1/private-alpha/library-tracks/similar`, body `{"tracks": [{"id": "libtr_..."}, ...]}`
(1-10 seeds). Finds tracks similar to the *mean* of the seeds, so it is a natural fit for a user's
taste profile (use several of their liked tracks as seeds). See
[the guide PDF](../guides/Find%20Similar%20Library%20Tracks%20(Multi-Track).pdf).


In [ ]:
def find_similar_multi(track_ids, limit=20, metadata_filter=None):
    # POST /private-alpha/library-tracks/similar -> {items: [{track, score}], pageInfo}
    body = {"tracks": [{"id": tid} for tid in track_ids]}
    if metadata_filter:
        body["metadataFilter"] = metadata_filter
    resp = session.post(
        f"{BASE_URL}/private-alpha/library-tracks/similar",
        params={"limit": limit}, json=body, timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

print("seeds:", SAMPLE_TRACK_IDS)
res = find_similar_multi(SAMPLE_TRACK_IDS, limit=10)
show_results(res)


## Putting it together: search -> tags -> explain (and listen)

Search for tracks, take the top result, fetch its tags, summarize them as an explanation, and
play it. This is the core of an explainable recommendation.


In [ ]:
res = search_by_prompt("warm acoustic folk, gentle fingerpicked guitar", limit=5)
top = res["items"][0]
track, score = top["track"], top.get("score")
print(f"top result: {track['id']}  |  {track.get('title')}  (score {score:.3f})")
print(summarize(get_model_outputs(track["id"], ["MainGenreV2", "MoodSimpleV2", "InstrumentsV2", "BpmV2", "CharacterV2"])))
play(track)


## Where to go next

- Build a **taste profile** from a user's likes (`data/users.csv`): resolve their Jamendo ids to
  Cyanite ids and feed several into multi-track similarity, or aggregate their tags to write a prompt.
- **Re-rank** similarity results for tag coherence and explain each pick via shared tags and `score`.
- Use **metadata filters** to steer along mood / genre / tempo axes.
- Cache model outputs locally and fetch each track once (shared rate limits / quotas).
